In [ ]:


const fs = await import("node:fs");

// Six-year graduation rates by entering freshman cohort
const years = [
  2010, 2011, 2012, 2013, 2014,
  2015, 2016, 2017, 2018, 2019
];

const series = [
  ["Berkeley", "#1f77b4",
    [89.3,89.2,88.4,88.3,88.6,88.7,89.4,88.3,88.1,90.1]],

  ["Davis", "#ff7f0e",
    [81.5,83.5,84.6,83.8,83.3,85.7,82.8,82.6,82.7,83.5]],

  ["Irvine", "#2ca02c",
    [86.9,83.4,82.5,85.0,83.8,82.8,86.1,85.2,85.7,85.7]],

  ["Los Angeles", "#d62728",
    [89.1,87.6,88.3,89.5,88.5,88.8,89.6,90.2,89.1,90.0]],

  ["Merced", "#9467bd",
    [68.8,65.3,69.1,70.7,71.3,72.1,70.0,71.1,72.3,70.0]],

  ["Riverside", "#8c564b",
    [75.6,76.9,78.1,77.9,79.4,78.0,77.2,78.6,79.2,78.7]],

  ["San Diego", "#e377c2",
    [86.3,84.2,85.5,86.0,84.1,88.1,85.0,84.3,83.2,82.6]],

  ["Santa Barbara", "#17becf",
    [81.2,80.8,81.9,81.9,82.9,84.2,82.9,82.7,82.4,84.3]],

  ["Santa Cruz", "#bcbd22",
    [79.1,78.6,74.7,74.1,77.0,80.5,80.8,74.6,76.0,77.3]]
];

// Chart dimensions
const W = 1100;
const H = 850;
const L = 75;
const R = 35;
const T = 35;
const B = 65;

const minY = 60;
const maxY = 95;

const x = i => L + i * (W-L-R) / (years.length-1);
const y = v => T + (maxY-v) * (H-T-B) / (maxY-minY);

// Start SVG
let svg = `
<svg xmlns="http://www.w3.org/2000/svg"
     viewBox="0 0 ${W} ${H}"
     width="${W}" height="${H}"
     style="font-family:Arial,sans-serif;background:white">
`;

// Horizontal gridlines
for (let v = 60; v <= 95; v += 5) {
  svg += `
    <line x1="${L}" y1="${y(v)}"
          x2="${W-R}" y2="${y(v)}"
          stroke="#dddddd" stroke-width="1"/>

    <text x="${L-15}" y="${y(v)+5}"
          text-anchor="end" font-size="16"
          fill="#555555">${v}%</text>
  `;
}

// Year labels
years.forEach((year, i) => {
  svg += `
    <text x="${x(i)}" y="${H-22}"
          text-anchor="middle" font-size="16"
          fill="#555555">${year}</text>
  `;
});

// Draw lines and points
series.forEach(([name, color, values]) => {
  const points = values
    .map((v,i) => `${x(i)},${y(v)}`)
    .join(" ");

  const lineWidth = name === "Santa Barbara" ? 5 : 3;

  svg += `
    <polyline points="${points}"
              fill="none"
              stroke="${color}"
              stroke-width="${lineWidth}"
              stroke-linejoin="round"
              stroke-linecap="round"/>
  `;

  values.forEach((v,i) => {
    svg += `
      <circle cx="${x(i)}" cy="${y(v)}"
              r="5" fill="${color}"
              stroke="white" stroke-width="1">
        <title>${name} — ${years[i]}: ${v}%</title>
      </circle>
    `;
  });
});

svg += "</svg>";

// Create legend
const legend = series.map(([name,color]) => `
  <span style="display:inline-flex;align-items:center;
               gap:7px;margin:6px 12px">
    <span style="display:inline-block;width:22px;
                 height:4px;background:${color}"></span>
    ${name}
  </span>
`).join("");

// Build complete HTML page
const html = `
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>UC Six-Year Graduation Rates</title>
<style>
  body {
    font-family: Arial, sans-serif;
    background: white;
    color: #222;
    margin: 30px;
  }
  h1 {
    text-align: center;
    font-size: 26px;
  }
  .subtitle {
    text-align: center;
    color: #666;
  }
  .chart {
    display: flex;
    justify-content: center;
    overflow-x: auto;
  }
  .legend {
    max-width: 1100px;
    margin: 15px auto;
    text-align: center;
    font-size: 14px;
  }
  .footer {
    text-align: center;
    font-size: 12px;
    color: #777;
    margin-top: 25px;
  }
</style>
</head>
<body>
<h1>Six-Year Graduation Rates Across UC Campuses</h1>
<p class="subtitle">2010–2019 Entering Freshman Cohorts</p>
<div class="chart">${svg}</div>
<div class="legend">${legend}</div>
<p class="footer">
Source: University of California Undergraduate Graduation Rates
dashboard. Values transcribed from provided screenshots.
Pell Grant filter not independently verified.
</p>
</body>
</html>
`;

// Save to the current project folder
fs.writeFileSync("uc_graduation_rates.html", html, "utf8");

console.log("Created uc_graduation_rates.html successfully.");